# CCAR-F — Claude Certified Architect – Foundations · Practice (scenario drills)

**Goal:** practice exam-style judgment on all six official CCAR-F scenarios, with instant feedback and links to the chapter section behind every item.

- Overview and study plan: [README.md](README.md) · Full mock exam: [02_homework.ipynb](02_homework.ipynb)
- Estimated time: 60–90 minutes (about 2–3 minutes per item, plus reading the feedback)
- Estimated API cost: **$0**. Everything runs offline; no API key is needed.
- Prerequisites: the chapters for the scenarios you drill (see the [task statement map](README.md#task-statement--chapter-map-all-30)), and the venv from [00-prerequisites](../../../00-prerequisites/README.md)

**How to use it**
1. Run the setup and helper cells once.
2. For each item, replace `None` with your letter (`"B"`) or, for *Select TWO* items, two letters (`"AD"`, `"A,D"` or `["A", "D"]`), then run the cell. You get feedback on each option you chose. Your **first** answer per item is what the report counts.
3. Work in **learning mode**: untimed, and reread the linked section whenever you miss one. Exam conditions are for the [mock](02_homework.ipynb).
4. Finish with the drill report and the mini-project (write your own drill).

All briefs and items are **original**. They test the same competencies as the official scenarios but are not the guide's sample questions or anything from the live exam. Answer keys are stored as SHA-256 hashes and the feedback notes are masked, so scrolling through the code won't give the answers away.

## Setup

In [ ]:
import base64
import hashlib
import json

try:
    from IPython.display import Markdown, display
except ImportError:  # plain Python: fall back to text
    Markdown = display = None

print("Offline drills: no API key or network needed.")

### Helpers (run, don't edit)

`drill(qid, answer)` checks an answer against the hashed key, shows feedback for every option you picked and links to the sections to review. `drill_report()` summarizes your first attempts by domain.

In [ ]:
DOMAINS = {1: 'Agentic Architecture & Orchestration', 2: 'Tool Design & MCP Integration', 3: 'Claude Code Configuration & Workflows', 4: 'Prompt Engineering & Structured Output', 5: 'Context Management & Reliability'}
META = {
 "s1q1": {
  "scenario": "S1",
  "domain": 5,
  "task": "5.2",
  "n": 1,
  "review": [
   [
    "Ch 9 §9.2",
    "../../09-escalation-human-in-the-loop/README.md#92-escalation-patterns"
   ],
   [
    "Ch 9 §9.1",
    "../../09-escalation-human-in-the-loop/README.md#91-when-to-escalate"
   ]
  ]
 },
 "s1q2": {
  "scenario": "S1",
  "domain": 5,
  "task": "5.2",
  "n": 1,
  "review": [
   [
    "Ch 9 §9.2",
    "../../09-escalation-human-in-the-loop/README.md#92-escalation-patterns"
   ]
  ]
 },
 "s1q3": {
  "scenario": "S1",
  "domain": 1,
  "task": "1.1",
  "n": 1,
  "review": [
   [
    "Ch 3 §3.1",
    "../../03-agent-sdk/README.md#31-the-agentic-loop"
   ],
   [
    "Ch 1 §1.3",
    "../../01-claude-api-fundamentals/README.md#13-stop_reason-why-the-model-stopped"
   ]
  ]
 },
 "s1q4": {
  "scenario": "S1",
  "domain": 1,
  "task": "1.5",
  "n": 2,
  "review": [
   [
    "Ch 3 §3.5",
    "../../03-agent-sdk/README.md#35-hooks-in-the-agent-sdk"
   ],
   [
    "Ch 9 §9.4",
    "../../09-escalation-human-in-the-loop/README.md#94-programmatic-enforcement-vs-prompt-guidance"
   ]
  ]
 },
 "s1q5": {
  "scenario": "S1",
  "domain": 5,
  "task": "5.1",
  "n": 1,
  "review": [
   [
    "Ch 11 §11.2",
    "../../11-context-management/README.md#112-trim-verbose-tool-results"
   ]
  ]
 },
 "s2q1": {
  "scenario": "S2",
  "domain": 1,
  "task": "1.7",
  "n": 1,
  "review": [
   [
    "Ch 11 §11.7",
    "../../11-context-management/README.md#117-sessions-resume-fork-or-start-fresh-task-17"
   ],
   [
    "Ch 5 §5.10",
    "../../05-claude-code/README.md#510-sessions-resume-name-fork"
   ]
  ]
 },
 "s2q2": {
  "scenario": "S2",
  "domain": 3,
  "task": "3.2",
  "n": 1,
  "review": [
   [
    "Ch 5 §5.5",
    "../../05-claude-code/README.md#55-skills-in-depth-skillmd-frontmatter"
   ],
   [
    "Ch 5 §5.2",
    "../../05-claude-code/README.md#52-path-imports-modular-claudemd"
   ]
  ]
 },
 "s2q3": {
  "scenario": "S2",
  "domain": 3,
  "task": "3.4",
  "n": 1,
  "review": [
   [
    "Ch 5 §5.6",
    "../../05-claude-code/README.md#56-plan-mode-vs-direct-execution"
   ]
  ]
 },
 "s2q4": {
  "scenario": "S2",
  "domain": 3,
  "task": "3.5",
  "n": 1,
  "review": [
   [
    "Ch 6 §6.9",
    "../../06-prompt-engineering/README.md#69-matching-the-strategy-to-the-task-and-iterating-expanded-beyond-the-guide"
   ]
  ]
 },
 "s2q5": {
  "scenario": "S2",
  "domain": 3,
  "task": "3.5",
  "n": 1,
  "review": [
   [
    "Ch 6 §6.4",
    "../../06-prompt-engineering/README.md#64-the-interview-pattern"
   ]
  ]
 },
 "s3q1": {
  "scenario": "S3",
  "domain": 1,
  "task": "1.2",
  "n": 1,
  "review": [
   [
    "Ch 3 §3.3",
    "../../03-agent-sdk/README.md#33-hub-and-spoke-coordinator-and-subagents"
   ],
   [
    "Ch 8 §8.2",
    "../../08-task-decomposition/README.md#82-dynamic-adaptive-decomposition"
   ]
  ]
 },
 "s3q2": {
  "scenario": "S3",
  "domain": 1,
  "task": "1.3",
  "n": 1,
  "review": [
   [
    "Ch 12 §12.1",
    "../../12-provenance/README.md#121-the-attribution-loss-problem"
   ],
   [
    "Ch 3 §3.4",
    "../../03-agent-sdk/README.md#34-the-subagent-tool-agent-formerly-task"
   ]
  ]
 },
 "s3q3": {
  "scenario": "S3",
  "domain": 5,
  "task": "5.3",
  "n": 2,
  "review": [
   [
    "Ch 10 §10.3",
    "../../10-multi-agent-error-handling/README.md#103-access-failure-vs-valid-empty-result"
   ],
   [
    "Ch 10 §10.6",
    "../../10-multi-agent-error-handling/README.md#106-the-structured-subagent-result"
   ]
  ]
 },
 "s3q4": {
  "scenario": "S3",
  "domain": 5,
  "task": "5.6",
  "n": 1,
  "review": [
   [
    "Ch 12 §12.4",
    "../../12-provenance/README.md#124-render-by-content-type"
   ]
  ]
 },
 "s3q5": {
  "scenario": "S3",
  "domain": 2,
  "task": "2.3",
  "n": 2,
  "review": [
   [
    "Ch 2 §2.2",
    "../../02-tool-use/README.md#22-tool-definition"
   ],
   [
    "Ch 3 §3.2",
    "../../03-agent-sdk/README.md#32-agentdefinition-configuration"
   ]
  ]
 },
 "s4q1": {
  "scenario": "S4",
  "domain": 2,
  "task": "2.5",
  "n": 1,
  "review": [
   [
    "Ch 13 §13.6",
    "../../13-claude-code-builtin-tools/README.md#136-incremental-investigation-grep-read-repeat"
   ],
   [
    "Ch 11 §11.5",
    "../../11-context-management/README.md#115-delegate-exploration-to-subagents-to-protect-the-coordinators-context"
   ]
  ]
 },
 "s4q2": {
  "scenario": "S4",
  "domain": 2,
  "task": "2.5",
  "n": 1,
  "review": [
   [
    "Ch 13 §13.4",
    "../../13-claude-code-builtin-tools/README.md#134-when-edit-fails-anchors-replace_all-and-the-read--write-fallback"
   ]
  ]
 },
 "s4q3": {
  "scenario": "S4",
  "domain": 2,
  "task": "2.5",
  "n": 1,
  "review": [
   [
    "Ch 13 §13.7",
    "../../13-claude-code-builtin-tools/README.md#137-tracing-through-wrapper-modules"
   ]
  ]
 },
 "s4q4": {
  "scenario": "S4",
  "domain": 2,
  "task": "2.4",
  "n": 1,
  "review": [
   [
    "Ch 13 §13.8",
    "../../13-claude-code-builtin-tools/README.md#138-built-in-tools-vs-mcp-tools"
   ],
   [
    "Ch 4 §4.2",
    "../../04-model-context-protocol/README.md#42-mcp-servers-clients-and-transports"
   ]
  ]
 },
 "s4q5": {
  "scenario": "S4",
  "domain": 5,
  "task": "5.4",
  "n": 1,
  "review": [
   [
    "Ch 11 §11.6",
    "../../11-context-management/README.md#116-structured-state-persistence-and-crash-recovery"
   ]
  ]
 },
 "s5q1": {
  "scenario": "S5",
  "domain": 4,
  "task": "4.5",
  "n": 2,
  "review": [
   [
    "Ch 7 §7.2",
    "../../07-message-batches/README.md#72-batch-vs-synchronous-matching-the-api-to-latency-requirements"
   ],
   [
    "Ch 7 §7.3",
    "../../07-message-batches/README.md#73-what-can-and-cannot-be-batched"
   ]
  ]
 },
 "s5q2": {
  "scenario": "S5",
  "domain": 4,
  "task": "4.5",
  "n": 1,
  "review": [
   [
    "Ch 7 §7.7",
    "../../07-message-batches/README.md#77-pre-flight-refine-on-a-sample-before-the-big-run"
   ]
  ]
 },
 "s5q3": {
  "scenario": "S5",
  "domain": 4,
  "task": "4.1",
  "n": 1,
  "review": [
   [
    "Ch 6 §6.2",
    "../../06-prompt-engineering/README.md#62-explicit-criteria-vs-vague-instructions"
   ]
  ]
 },
 "s5q4": {
  "scenario": "S5",
  "domain": 3,
  "task": "3.6",
  "n": 1,
  "review": [
   [
    "Ch 5 §5.9",
    "../../05-claude-code/README.md#59-claude-code-in-cicd-headless-mode"
   ]
  ]
 },
 "s5q5": {
  "scenario": "S5",
  "domain": 4,
  "task": "4.6",
  "n": 1,
  "review": [
   [
    "Ch 8 §8.4",
    "../../08-task-decomposition/README.md#84-independent-review-instances-why-self-review-is-weak"
   ]
  ]
 },
 "s6q1": {
  "scenario": "S6",
  "domain": 4,
  "task": "4.3",
  "n": 1,
  "review": [
   [
    "Ch 2 §2.4",
    "../../02-tool-use/README.md#24-json-schemas-for-structured-output"
   ]
  ]
 },
 "s6q2": {
  "scenario": "S6",
  "domain": 4,
  "task": "4.4",
  "n": 2,
  "review": [
   [
    "Ch 6 §6.6",
    "../../06-prompt-engineering/README.md#66-self-correction"
   ],
   [
    "Ch 6 §6.5",
    "../../06-prompt-engineering/README.md#65-validation-and-retry-with-feedback"
   ],
   [
    "Ch 2 §2.5",
    "../../02-tool-use/README.md#25-syntax-vs-semantic-errors"
   ]
  ]
 },
 "s6q3": {
  "scenario": "S6",
  "domain": 4,
  "task": "4.4",
  "n": 1,
  "review": [
   [
    "Ch 6 §6.5",
    "../../06-prompt-engineering/README.md#65-validation-and-retry-with-feedback"
   ],
   [
    "Ch 9 §9.9",
    "../../09-escalation-human-in-the-loop/README.md#99-routing-review-work-with-limited-capacity"
   ]
  ]
 },
 "s6q4": {
  "scenario": "S6",
  "domain": 5,
  "task": "5.5",
  "n": 1,
  "review": [
   [
    "Ch 9 §9.8",
    "../../09-escalation-human-in-the-loop/README.md#98-segment-accuracy-and-stratified-sampling"
   ]
  ]
 },
 "s6q5": {
  "scenario": "S6",
  "domain": 5,
  "task": "5.5",
  "n": 1,
  "review": [
   [
    "Ch 9 §9.8",
    "../../09-escalation-human-in-the-loop/README.md#98-segment-accuracy-and-stratified-sampling"
   ],
   [
    "Ch 9 §9.7",
    "../../09-escalation-human-in-the-loop/README.md#97-confidence-calibration"
   ]
  ]
 }
}
KEY_HASH = {
 "s1q1": "0cd35a1ff1fd38484b873f1299c5ff054a824b25d9378bf22e4ea6152e5d10d8",
 "s1q2": "584c9e9150066c481e3180f1b4fda58b771f98c2583c752a1004cacfeceb9985",
 "s1q3": "9955e90149036b4060c1b37dcbbf869e4d0517ff44d9ba4efdb59ca836064ad7",
 "s1q4": "a7933c72646cd1ec8aaaeb1454c43e70892918ec4691c47115728ccf0a748f1e",
 "s1q5": "8006b5ddb5cdfc51b2c30167545c8f93dda0940c12c08f190c27a0a0f8170b1c",
 "s2q1": "31c004376d9187d4f9433ea90f59eb708d98ad510a9f09d731ec86ade1b60e9e",
 "s2q2": "ba1f752c73e821ef1543551388d3e10b28931f4bd78d3cedcabe50b6f982fad2",
 "s2q3": "ee79e896fdfb1bd035452b1d4d6103fa0cfde13d37502d6de6c98f3f9c55c442",
 "s2q4": "b4e7efb78281fa11faae2bf7c715da273aa8bc40fe9434f23b136f1b55e67fc3",
 "s2q5": "3d6d2107775d132ec6de1544c5130f654e2c29363ad9b162b9165907aa3bdfe7",
 "s3q1": "07f695dc1f3e39a633a53f86381db18861d0afcdea9fa4fce3d43b4fab31060b",
 "s3q2": "fe2dd80fcb7c4fd3fdf793a9c467201329b3178ea9c964538a65a5faf3289ffb",
 "s3q3": "13b688e974084ab99bda6341f177be7b6d4b2c98f71da58f33f22756e7faa60f",
 "s3q4": "ed12c07723d209ab3a2e27b01e6b3d81e3bdc0e13feb3727f099631e107050be",
 "s3q5": "42f8c7b9028da41f57e1a0e36c7fdb5475f196afacd0dfed32efa9ab1649f030",
 "s4q1": "63ebac06d084072319d6c84861b1c36d55fe98d19043728ae86a4d447189f222",
 "s4q2": "414943651ce8516b3908f04411157638767acb8e3805c7acd8ed26157e07c150",
 "s4q3": "b50c6d27232d9dbc5f89824e5c4e4f824433b7984a167d4b7a474e0052c61ce6",
 "s4q4": "747d5d0ec5b70e3deb192c46b3f8e2fc1bcbd09e8b8b7097f42e4e093358c894",
 "s4q5": "ef93bf334367dea377a43c24f1af3e22436f0dbcaeaf56f630ed70f889ea66a8",
 "s5q1": "60e086591f73909ad2c174cc6311b248c4fc717df481738951699f79410b4479",
 "s5q2": "a5f178ef8e80428ca3cf16d406c0685baa0bfa15101df4c3e4bde14da2d395fb",
 "s5q3": "a48b7c9c53a247060c8f2a65d45eb2b8288b4472baba73b9745cef65b1bb4f3d",
 "s5q4": "016a7cf453c813fca552befc81c37427e563935524317efef73c0e222cc70c8d",
 "s5q5": "e5030bc7ac145bc017a5d9e3b8cd23a7504cdef3154981799a4929fdfe296b52",
 "s6q1": "966f83336ff0082f72fe1cbe5759c7ae34d4ee7e37ae4f8f9d3f608aaec4d00d",
 "s6q2": "49fb787050832277683141e9c3be4c2efbf2d5939868da9af5505a1f8e249d56",
 "s6q3": "215c03b83e586673f73f59484fbfe2989b67a1ac358dfe1df06d786a22eb05ce",
 "s6q4": "5ed5aa39ea2128fbf78261fb85be3181bdd2c96a87285dafb84965bf19db3e1c",
 "s6q5": "3e5cbd8b645f7ef848b9dce47bc96cb675e34f4d74ea3072bffe10ae13ebd109"
}
NOTES = {
 "..."
}
FIRST: dict[str, bool] = {}   # first attempt per item (what the report counts)
LAST: dict[str, bool] = {}


def _h(qid: str, letters: str) -> str:
    return hashlib.sha256(f"{qid}:{letters}".encode()).hexdigest()


def _show(md: str) -> None:
    if display is not None:
        display(Markdown(md))
    else:
        print(md)


def _note(qid: str, letter: str) -> str | None:
    """Decode the feedback note for one option (notes are masked so the code does not give answers away)."""
    blob = NOTES.get(_h(qid, letter))
    if blob is None:
        return None
    data = base64.b64decode(blob)
    key = hashlib.sha256(f"note:{qid}:{letter}".encode()).digest()
    stream = (key * (len(data) // len(key) + 1))[: len(data)]
    return bytes(a ^ b for a, b in zip(data, stream)).decode()


def _letters(answer) -> str | None:
    """Accept "b", "A,D", "a d", ["A", "D"] or {"D", "A"}; return sorted unique letters, or None."""
    if answer is None:
        return None
    if isinstance(answer, (list, tuple, set)):
        answer = "".join(str(a) for a in answer)
    if not isinstance(answer, str):
        return None
    letters = sorted({c for c in answer.upper() if c.isalpha()})
    return "".join(letters) or None


def drill(qid: str, answer) -> bool | None:
    meta = META.get(qid)
    if meta is None:
        print(f"Unknown item id {qid!r}.")
        return None
    need = meta["n"]
    letters = _letters(answer)
    if letters is None:
        print(f"{qid}: replace None with your answer" + (f" ({need} letters, e.g. \"AC\")." if need > 1 else " (one letter, e.g. \"B\")."))
        return None
    if any(c not in "ABCDE" for c in letters):
        print(f"{qid}: use option letters A–E only (you gave {answer!r}).")
        return None
    ok = len(letters) == need and _h(qid, letters) == KEY_HASH[qid]
    FIRST.setdefault(qid, ok)
    LAST[qid] = ok
    lines = [f"**{'✓ Correct' if ok else '✗ Not quite'}** · {qid.upper()} · Domain {meta['domain']} ({DOMAINS[meta['domain']]}) · Task {meta['task']}"]
    if len(letters) != need:
        lines.append(f"This item asks you to select **{need}**; you selected {len(letters)}.")
    for L in letters:
        note = _note(qid, L)
        if note:
            lines.append(f"- **{L}**: {note}")
    if not ok:
        lines.append("Think it through again before you retry; the report keeps your first attempt.")
    lines.append("Review: " + ", ".join(f"[{label}]({url})" for label, url in meta["review"]))
    _show("\n\n".join(lines))
    return ok


def drill_report() -> dict:
    """First-attempt accuracy by domain and scenario, plus the sections to reread."""
    if not FIRST:
        print("No drills answered yet. Answer some items above, then run this again.")
        return {}
    by_dom: dict[int, list[bool]] = {}
    by_sc: dict[str, list[bool]] = {}
    for qid, ok in FIRST.items():
        by_dom.setdefault(META[qid]["domain"], []).append(ok)
        by_sc.setdefault(META[qid]["scenario"], []).append(ok)
    total = sum(FIRST.values())
    print(f"Drills answered: {len(FIRST)}/{len(META)} · first-attempt accuracy {total}/{len(FIRST)} = {total / len(FIRST):.0%}\n")
    for d in sorted(by_dom):
        r = by_dom[d]
        print(f"  Domain {d} {DOMAINS[d]:<40} {sum(r)}/{len(r)}  {'█' * sum(r)}{'░' * (len(r) - sum(r))}")
    print("  " + "  ".join(f"{sc}: {sum(r)}/{len(r)}" for sc, r in sorted(by_sc.items())))
    missed = [q for q, ok in FIRST.items() if not ok]
    if missed:
        md = ["**Reread before the mock:**"] + [
            f"- {q.upper()} (task {META[q]['task']}): " + ", ".join(f"[{lb}]({u})" for lb, u in META[q]["review"]) for q in missed]
        _show("\n".join(md))
    else:
        print("\nNo misses on first attempts. Time for the mock exam.")
    return {"answered": len(FIRST), "correct": total, "by_domain": {d: (sum(r), len(r)) for d, r in by_dom.items()}}

## 1 · S1: Customer Support Resolution Agent (official scenario SC1)

*Primary domains: D1 · D2 · D5*

**Harbor & Pine** is a fictional online furniture retailer. Its support agent runs on the **Claude Agent SDK** and uses four custom MCP tools: `get_customer`, `lookup_order`, `process_refund` and `escalate_to_human`. Refunds above **$500** need a human. Requests are messy: late deliveries, damaged sofas, partial refunds, account mix-ups. The goal is high first-contact resolution **and** escalating the right cases.

**S1.1.** A customer writes: 'This is ridiculous, my sofa is three weeks late!' The agent can expedite a replacement shipment itself. The customer has not asked for a human. What should the agent do?

- A) Skip the emotional part and give the tracking details, since customers mainly want facts in such cases.
- B) Acknowledge the frustration, offer the expedited shipment, and escalate only if the customer then asks for a person.
- C) Escalate immediately, because strongly negative sentiment means the case is now too delicate for an automated agent.
- D) Ask the customer to calm down before continuing, so the conversation can stay focused on the facts.

In [ ]:
drill("s1q1", None)  # replace None with your answer, e.g. "B"

**S1.2.** A customer received a Harbor & Pine marketing email with a 15% code, sent by mistake two days after the promotion ended, and asks the agent to honor it. The pricing policy covers price drops within 14 days of purchase but says nothing about promotional emails sent in error. What should the agent do?

- A) Apply the 14-day price-drop rule by analogy and refund 15% if the purchase was within 14 days.
- B) Honor the code as a goodwill gesture, since the mistake was the company's and the amount is small.
- C) Decline politely, because the promotion had ended before the email was sent to the customer.
- D) Escalate to a human, because the policy is silent on codes the company itself sent in error.

In [ ]:
drill("s1q2", None)  # replace None with your answer, e.g. "B"

**S1.3.** The agent loop stops after a hard cap of 5 iterations, which the team treats as the main stopping rule. Complex return cases that need 7 tool calls get cut off mid-investigation. What is the right design?

- A) Raise the cap to 7, which covers the most complex return cases the team has seen in production so far.
- B) Stop as soon as a response contains any text block, since a text block means Claude is addressing the customer.
- C) Stop on `stop_reason == "end_turn"`, and keep a much higher cap only as a safety guard against runaway loops.
- D) Stop when the assistant's text includes words such as 'resolved' or 'refund processed'.

In [ ]:
drill("s1q3", None)  # replace None with your answer, e.g. "B"

**S1.4.** *(Select TWO)* The team is moving rules out of the system prompt. Which TWO of these rules should be enforced with hooks rather than left to prompt instructions?

- A) Refunds above $500 must go to a human instead of `process_refund`.
- B) Try to resolve the issue before offering a human, unless the customer asks for one.
- C) `process_refund` may only run once `get_customer` has confirmed who the customer is.
- D) Keep a warm, apologetic tone when a delivery is late.
- E) End each conversation with a short summary of what was agreed.

In [ ]:
drill("s1q4", None)  # replace None with your answer, e.g. "AC"

**S1.5.** `lookup_order` returns about 60 fields per order (warehouse codes, carrier internals, tax lines). After a few lookups, the agent forgets details the customer gave earlier. What is the best fix?

- A) Trim each lookup result to the fields relevant to returns and refunds before it enters the context.
- B) Use a model with a larger context window, so the full 60-field results and the conversation both fit.
- C) Ask the model to ignore irrelevant fields when it reads tool results, in the system prompt.
- D) Summarize the whole conversation after every tool call, so the context never grows too large.

In [ ]:
drill("s1q5", None)  # replace None with your answer, e.g. "B"

## 2 · S2: Code Generation with Claude Code (official scenario SC2)

*Primary domains: D3 · D5*

A fintech team maintains a **Django** monolith and a **React** front end. Twelve engineers use **Claude Code** for features, refactors, debugging and docs. The repo has a long project `CLAUDE.md`, a few skills, and a habit of very long sessions. You are asked to make the setup predictable for everyone.

**S2.1.** An engineer wants to continue a week-old session that mapped the payments module. Since then, the team has refactored most of the files it analyzed. What is the most reliable way to continue?

- A) Start a new session and give it a short structured summary of the findings that are still valid.
- B) Resume the old session by name, since it already holds a detailed map of the payments module.
- C) Resume the old session and ask Claude to re-check everything it found last week, file by file.
- D) Fork the old session, so the stale analysis is preserved in the original for later comparison.

In [ ]:
drill("s2q1", None)  # replace None with your answer, e.g. "B"

**S2.2.** The project `CLAUDE.md` is 350 lines. Coding standards apply all the time, but a 120-line release runbook matters only during releases. What should you do?

- A) Leave it as is, because Claude Code automatically skips sections that are irrelevant to the task.
- B) Keep the universal standards in `CLAUDE.md` and move the release runbook into a project skill.
- C) Split the runbook into a separate file and `@`-import it from `CLAUDE.md` to keep things tidy.
- D) Move the runbook into a path-scoped rule that matches `release/**`, so it loads only when needed.

In [ ]:
drill("s2q2", None)  # replace None with your answer, e.g. "B"

**S2.3.** The team must upgrade from Django 4.2 to 5.x; about 50 files are affected and some APIs changed. Which workflow fits best?

- A) Stay in plan mode for the whole upgrade, so no file is changed without careful thought.
- B) Start in direct execution and switch to plan mode only if unexpected problems appear.
- C) Use plan mode to investigate and agree on a migration plan, then execute the approved plan directly.
- D) Open 50 separate sessions, one per affected file, so each change gets a clean and fully focused context window.

In [ ]:
drill("s2q3", None)  # replace None with your answer, e.g. "B"

**S2.4.** A bug report describes two problems in the payment summary: tax is rounded before currency conversion, and conversion uses the wrong rate date. Fixing one changes the expected output of the other. How should the engineer brief Claude?

- A) Only the rate-date problem, since fixing it will probably make the rounding problem disappear.
- B) Ask Claude to find all problems in the payment module on its own, without describing them.
- C) In one detailed message that describes both problems and how they interact.
- D) One problem at a time, starting with the rounding bug, then the rate date in a later prompt.

In [ ]:
drill("s2q4", None)  # replace None with your answer, e.g. "B"

**S2.5.** An engineer must add a caching layer to the pricing service, a domain they barely know. What is a good first step with Claude Code?

- A) Ask Claude to implement a standard Redis cache right away, then fix any problems as they show up in testing.
- B) Ask Claude to interview them first about invalidation, consistency needs and failure modes, then implement.
- C) Copy the caching layer from another service and ask Claude to adapt the names to this one.
- D) Write a detailed specification alone first, then hand it to Claude to implement in one go.

In [ ]:
drill("s2q5", None)  # replace None with your answer, e.g. "B"

## 3 · S3: Multi-Agent Research System (official scenario SC3)

*Primary domains: D1 · D2 · D5*

A public-health nonprofit runs a **multi-agent literature review** built on the Claude Agent SDK. A coordinator delegates to a web-search subagent, a paper-analysis subagent, a synthesis subagent and a report subagent. Reports go to policymakers, so every claim must be traceable to a source.

**S3.1.** A review on 'health effects of urban heat' must cover outdoor workers, a group the client named. The plan assigned that subtopic, but the synthesis draft notes that only one weak source turned up on it, while every other section is well sourced. The report subagent has not run yet. What should the coordinator do next?

- A) Ask the synthesis subagent to strengthen the outdoor-worker section from its general medical knowledge.
- B) Send targeted queries on outdoor workers to the search and paper-analysis subagents, then re-run synthesis with the new findings; if the gap remains, mark it in the report.
- C) Re-run the whole pipeline from the start with the same plan, in the hope that the searches return more this time.
- D) Pass the draft to the report subagent as it is, because the plan already assigned the subtopic and the subagents did their part.

In [ ]:
drill("s3q1", None)  # replace None with your answer, e.g. "B"

**S3.2.** The coordinator passes paper-analysis results to the synthesis subagent as one paragraph of prose. Citations in the final report are often wrong or missing. What should it pass instead?

- A) The same prose with a numbered bibliography appended at the end of the paragraph.
- B) A shorter summary of the analysis, so the synthesis subagent has more room for its own reasoning.
- C) Structured records per finding: claim, excerpt, document name, page, URL and publication date.
- D) Only the URLs of the analyzed papers, so the synthesis subagent can re-read them itself.

In [ ]:
drill("s3q2", None)  # replace None with your answer, e.g. "B"

**S3.3.** *(Select TWO)* In one run, the paper-analysis subagent's query for 'heat and sleep quality in adolescents' ran normally and matched no studies, while a second query to another journal database timed out on every attempt. Which TWO statements describe how the subagent should report these two outcomes?

- A) Report both as empty lists, since in both cases the coordinator receives no studies to analyze.
- B) The timed-out query is an access failure: report it as an error with its type and what was attempted, not as zero studies.
- C) Stop the whole review, because a run with a failed query cannot produce a trustworthy report.
- D) The query that ran and matched nothing is a successful result with no studies: 'no evidence found' is itself a finding.
- E) Report both as errors, so the coordinator retries until each query returns at least one study.

In [ ]:
drill("s3q3", None)  # replace None with your answer, e.g. "AC"

**S3.4.** The report converts everything to prose, including a 12-row comparison of hospital admission rates by city and year, which readers find hard to use. What should change?

- A) Convert the whole report to bullet lists, which are much easier to scan than prose for busy policy readers.
- B) Render by content type: numeric comparisons as tables, narrative developments as prose, findings as lists.
- C) Ask the report subagent to write shorter paragraphs with more frequent headings.
- D) Move all numbers to an appendix so the main text reads smoothly for policymakers.

In [ ]:
drill("s3q4", None)  # replace None with your answer, e.g. "B"

**S3.5.** *(Select TWO)* The synthesis subagent was given 16 tools, including full web search, and it often launches new literature searches instead of synthesizing. Almost all of its genuine lookups are small: confirming a study's publication year or the size of its cohort. Which TWO changes are best?

- A) Cut the synthesis subagent's toolset down to what synthesis actually needs.
- B) Remove the synthesis subagent and let the coordinator write the synthesis itself.
- C) Add a narrow `get_citation_metadata` tool for those small checks; open-ended research still goes via the coordinator.
- D) Give it every tool the coordinator has, so it never needs to hand any work back to the coordinator.
- E) Add 'do not start new literature searches' to its prompt and keep all 16 tools available to it.

In [ ]:
drill("s3q5", None)  # replace None with your answer, e.g. "AC"

## 4 · S4: Developer Productivity with Claude (official scenario SC4)

*Primary domains: D2 · D3 · D1*

A platform team built an internal **"repo guide"** agent on the Claude Agent SDK. It helps engineers explore unfamiliar services, understand legacy code and automate chores, using the built-in tools (`Read`, `Write`, `Edit`, `Bash`, `Grep`, `Glob`) plus MCP servers for internal docs and code search across 40 repositories.

**S4.1.** An engineer asks the agent: 'How does the refund flow work in the `orders` service?' The service has about 2,000 files. How should the agent start?

- A) `Grep` for refund entry points (routes, handlers), then `Read` those files and follow their imports.
- B) `Read` every file in the service first, so that the explanation is based on the complete, current code.
- C) Run `Bash` with `find . -type f | xargs cat` to get all the code into one tool result.
- D) `Glob` for `**/*refund*` and explain the flow from the file names alone.

In [ ]:
drill("s4q1", None)  # replace None with your answer, e.g. "B"

**S4.2.** The agent tries to change one `return None` in `billing/utils.py`, but `Edit` fails because that exact text appears six times in the file. Which fallback is reliable?

- A) `Read` the full file, then `Write` it back with the one intended change.
- B) Use `Bash` to run `sed -i 's/return None/return 0/' billing/utils.py` instead.
- C) Use `Glob` to find the file again, then repeat the `Edit` call.
- D) Retry the same `Edit` call; edits often succeed on a second attempt.

In [ ]:
drill("s4q2", None)  # replace None with your answer, e.g. "B"

**S4.3.** `payments/index.ts` re-exports helpers from several files, some under new names (`export { charge as createCharge } ...`). You need every usage of these helpers across the repo. How should the agent search?

- A) First list every name the wrapper exports, then `Grep` for each name across the codebase.
- B) `Glob` for `**/payments/**` and read every file it returns.
- C) `Grep` for the original function names only, since renamed exports refer to the same code.
- D) `Grep` only for `payments/index` to find the files that import the wrapper module.

In [ ]:
drill("s4q3", None)  # replace None with your answer, e.g. "B"

**S4.4.** The internal `code_search` MCP tool indexes all 40 repositories with semantic search, but the agent keeps using `Grep`, which only sees the current checkout. What should you do first?

- A) Remove `Grep` from the agent's tools so it has no choice but to use `code_search`.
- B) Add a `PreToolUse` hook that silently rewrites every `Grep` call into an equivalent `code_search` call.
- C) Add 'always use code_search' to the system prompt, near the top.
- D) Rewrite the `code_search` description: what it covers, what it returns, and when to prefer it over `Grep`.

In [ ]:
drill("s4q4", None)  # replace None with your answer, e.g. "B"

**S4.5.** A multi-agent modernization analysis runs for hours across several services. When the process crashes, all progress is lost and the run starts over. What design fixes this?

- A) Resume the crashed coordinator session with `--resume` and trust it to pick up exactly where all the agents stopped.
- B) Each agent exports its state to a known location; on restart the coordinator loads a manifest and injects it into prompts.
- C) Lower the per-agent token budget, so each run is short enough to finish before a crash.
- D) Run the whole analysis in a single very long session, so there is only one thing to restart.

In [ ]:
drill("s4q5", None)  # replace None with your answer, e.g. "B"

## 5 · S5: Claude Code for Continuous Integration (official scenario SC5)

*Primary domains: D3 · D4*

A payments platform runs **Claude Code in CI** on every merge request: automated review, test generation and nightly maintenance jobs. Leadership wants lower cost, fewer false positives and no blocked pipelines.

**S5.1.** *(Select TWO)* Which TWO workloads are good candidates for the Message Batches API?

- A) The blocking review that must pass before a merge request can merge.
- B) An agent that calls the team's own client tools, run by its code, and continues with their results.
- C) Nightly regeneration of API reference docs for all services.
- D) An interactive assistant that answers engineers' questions in chat.
- E) A weekly analysis of test-coverage gaps across the monorepo.

In [ ]:
drill("s5q1", None)  # replace None with your answer, e.g. "AC"

**S5.2.** You are about to send 5,000 legacy files through a batch job that generates documentation. What should you do first?

- A) Split the 5,000 into 50 batches and submit them all in parallel to finish faster.
- B) Submit all 5,000 files at once; the 50% batch discount makes a second full run affordable if needed.
- C) Use the synchronous API for the first 1,000 files to compare speed with batches.
- D) Refine the prompt on a small sample of files until first-pass results are good, then run the batch.

In [ ]:
drill("s5q2", None)  # replace None with your answer, e.g. "B"

**S5.3.** The review labels the same SQL string concatenation 'critical' in one merge request and 'low' in another. What is the most effective fix?

- A) Drop severities and post every finding at the same level, so there is nothing to disagree on.
- B) Tell the reviewer to be consistent with the severities it assigned in earlier merge requests.
- C) Ask the reviewer to report its confidence in each severity and post only high-confidence ones.
- D) Define each severity level with explicit criteria and a concrete code example for each level.

In [ ]:
drill("s5q3", None)  # replace None with your answer, e.g. "B"

**S5.4.** Generated tests are mostly low-value (getters, trivial constructors) and rarely cover error paths. What is the most effective fix?

- A) Document in `CLAUDE.md` what a valuable test is for this codebase, with the fixtures to use.
- B) Run test generation through the Message Batches API.
- C) Ask for twice as many tests per changed file, so the important cases are more likely to appear among them.
- D) Add 'write good tests' to the CI prompt, with emphasis.

In [ ]:
drill("s5q4", None)  # replace None with your answer, e.g. "B"

**S5.5.** Code is generated by one Claude Code step and reviewed in the same session with the prompt 'Now review your code carefully, step by step.' Few bugs are found. What is the best change?

- A) Make the review prompt stricter: 'Assume there are at least three bugs and find them.'
- B) Enable extended thinking in the same session for the review step.
- C) Review twice in the same session and report only issues found both times.
- D) Review in a separate instance that sees the code but not the generator's reasoning.

In [ ]:
drill("s5q5", None)  # replace None with your answer, e.g. "B"

## 6 · S6: Structured Data Extraction (official scenario SC6)

*Primary domains: D4 · D5*

An insurer's claims-intake system uses Claude to extract fields from **repair invoices, medical bills and police reports** (PDF and scans) into JSON for the claims platform. Output is validated against a JSON schema; downstream systems reject malformed records. Accuracy matters more than speed.

**S6.1.** Some repair invoices carry no policy number, but the extraction schema marks `policy_number` as a required string. The model sometimes invents plausible policy numbers. What is the fix?

- A) Add 'never invent a policy number' to the prompt, and keep the field a required string in the schema.
- B) Validate the format with a regex and retry until the number passes the check.
- C) Make `policy_number` nullable so the model can return `null` when the document does not contain it.
- D) Have the model fill a placeholder such as 'PN-0000000' that passes the policy-number format check.

In [ ]:
drill("s6q1", None)  # replace None with your answer, e.g. "B"

**S6.2.** *(Select TWO)* Extraction uses tool use with a strict JSON schema, yet some extracted invoices have line items that do not sum to the stated total. Spot checks show the invoices themselves add up: the model skipped or misread lines. Which TWO changes address this?

- A) Extract `calculated_total` next to `stated_total`, recompute in code, and set `conflict_detected` on a mismatch.
- B) Switch from tool use to plain JSON in the text, which the model can format more flexibly.
- C) Make the JSON schema stricter, with more required fields, tighter types and minimum lengths for every array.
- D) On a mismatch, retry with the document, the failed extraction and the specific error.
- E) Add 'make sure the line items add up to the total' to the prompt and accept whatever comes back.

In [ ]:
drill("s6q2", None)  # replace None with your answer, e.g. "AC"

**S6.3.** A police report never states the date of the incident; it only says 'last Tuesday'. Validation fails on `incident_date`. What should happen?

- A) Compute the date from the report's filing date, assuming that 'last Tuesday' is relative to the filing.
- B) Drop the report from the pipeline, since incomplete documents are not usable.
- C) Return `null` (or the raw text in a separate field) and route the claim for human review, instead of retrying.
- D) Retry with the validation error until the model returns a valid ISO date.

In [ ]:
drill("s6q3", None)  # replace None with your answer, e.g. "B"

**S6.4.** Overall field accuracy is 97%, and leadership wants to stop human review. A breakdown shows 78% accuracy on handwritten police reports. What should you do?

- A) Stop human review for all documents, because 97% overall clears the agreed automation threshold.
- B) Validate accuracy by document type and field, and keep review where segments fall short.
- C) Raise the confidence threshold for all documents equally.
- D) Stop review for all types and add a disclaimer to police-report claims.

In [ ]:
drill("s6q4", None)  # replace None with your answer, e.g. "B"

**S6.5.** High-confidence extractions now skip human review. How do you keep measuring their error rate and catch new error patterns?

- A) Trust the confidence scores; high-confidence extractions have already proven correct in validation.
- B) Re-run each high-confidence extraction and compare the two outputs.
- C) Review only the extractions that downstream systems reject.
- D) Stratified random sampling of high-confidence extractions across document types, reviewed by humans.

In [ ]:
drill("s6q5", None)  # replace None with your answer, e.g. "B"

## 7 · Your drill report

Run this after the drills. It counts your **first** attempt per item, breaks accuracy down by domain and scenario, and lists the sections to reread. If a domain is below about 80%, go back to its week in the [study plan](README.md#8-week-study-plan-using-this-repo) before you take the mock.

In [ ]:
report = drill_report()

## 8 · Mini-project: write one drill of your own

Writing a good item is the fastest way to learn what the exam rewards. Pick the task statement where you are weakest and write one **original** item for it: a short situation, one best answer, and three distractors that are plausible anti-patterns (see the [trap patterns](README.md#trap-patterns-principles-from-the-official-sample-questions)). Give each option a one-line reason.

`check_item()` reviews the structure offline:
- 4 options for a single-answer item (or 5 options with exactly 2 correct for *Select TWO*);
- every option has a reason;
- the correct option is **not** the longest one (test-takers learn to spot "longest = right");
- no "all of the above" or "none of the above";
- the stem ends with a question.

> **Try it:** write an item for task 1.5 or 2.5, run `check_item(MY_ITEM)`, then swap it with a study partner and see whether they pick your intended answer.

In [ ]:
def check_item(item: dict | None) -> bool:
    """Structural review of a self-written drill. Prints problems; never raises."""
    try:
        if not item or not item.get("stem") or not item.get("options"):
            print("Fill in MY_ITEM first: a stem and a list of options.")
            return False
        problems = []
        opts = item["options"]
        correct = [o for o in opts if o.get("correct")]
        if len(correct) == 1 and len(opts) != 4:
            problems.append("A single-answer item should have exactly 4 options.")
        elif len(correct) == 2 and len(opts) != 5:
            problems.append("A Select TWO item should have 5 options.")
        elif len(correct) not in (1, 2):
            problems.append(f"Mark 1 (or 2 for Select TWO) options as correct; you marked {len(correct)}.")
        if any(not (o.get("why") or "").strip() for o in opts):
            problems.append("Every option needs a one-line reason ('why').")
        if any(not (o.get("text") or "").strip() for o in opts):
            problems.append("Every option needs text.")
        lengths = [len(o.get("text") or "") for o in opts]
        if correct and len(correct) == 1 and max(lengths) == len(correct[0].get("text") or "") and lengths.count(max(lengths)) == 1:
            problems.append("The correct option is the longest one; lengthen a distractor or tighten the answer.")
        if any(p in (o.get("text") or "").lower() for o in opts for p in ("all of the above", "none of the above")):
            problems.append("Avoid 'all/none of the above'.")
        if not item["stem"].rstrip().endswith("?"):
            problems.append("End the stem with a clear question.")
        if problems:
            print("Revise:\n- " + "\n- ".join(problems))
            return False
        print("✓ Structure looks good. Now check the content: is exactly one answer defensible under the stated constraints?")
        return True
    except Exception as err:  # keep the notebook running whatever the input looks like
        print(f"Could not check the item: {err!r}")
        return False


MY_ITEM = {
    "task": "1.5",                      # the task statement you target
    "stem": None,                        # TODO: a short situation that ends with a question
    "options": [                         # TODO: 4 options (5 for Select TWO); mark the correct one(s)
        # {"text": "...", "correct": True,  "why": "..."},
        # {"text": "...", "correct": False, "why": "..."},
    ],
}
check_item(MY_ITEM)

<details><summary>Reference: an example item that passes <code>check_item()</code></summary>

```python
EXAMPLE = {
    "task": "1.5",
    "stem": "Your support agent must never issue a refund to an account flagged for fraud review. "
            "Today the rule is in the system prompt, and logs show two violations last month. What should you add?",
    "options": [
        {"text": "A PreToolUse hook on process_refund that checks the fraud flag and denies the call with a reason.",
         "correct": True, "why": "Deterministic: the hook runs on every refund call."},
        {"text": "A second, more forceful sentence about fraud-flagged accounts at the very top of the system prompt.",
         "correct": False, "why": "Still probabilistic."},
        {"text": "Few-shot examples that show the agent refusing refunds for flagged accounts in several situations.",
         "correct": False, "why": "Improves consistency but cannot guarantee compliance."},
        {"text": "A weekly report of refunds issued to flagged accounts, reviewed by the fraud team.",
         "correct": False, "why": "Detects violations after the money has moved."},
    ],
}
check_item(EXAMPLE)
```
</details>

## Recap

- Each official scenario mixes domains. Read the brief for its **constraints** (targets, thresholds, who can see what) before you look at the options.
- The recurring winners: guarantees in code, the smallest fix that addresses the root cause, visible and specific failures, least privilege, isolation of verbose work, and honest handling of uncertainty (escalate, annotate, return `null`).
- The recurring losers: stronger wording in prompts for hard rules, invented mechanisms, over-engineered first steps, bigger models or windows, and silent suppression.
- Your drill report tells you which weeks of the [study plan](README.md#8-week-study-plan-using-this-repo) to revisit.

Next: [the full-length mock exam](02_homework.ipynb), under exam conditions.